# Dental AI — Variant 1: Plný tréning (YOLOv8s, mega-dataset)

- Dataset: `eriksmite/mega-dataset-dental` (47 797 anotácií, 5 tried, ~765 MB)
- Model: YOLOv8s, imgsz 640, 60 epók
- Očakávaný čas: ~5–7 h na P100/T4 → zmestí sa do 1 dňa
- Výstup: `best.pt` ako kernel output

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print(ultralytics.__version__, torch.__version__, torch.cuda.is_available())
# Disk check — aby sa nezopakoval problem s miestom
!df -h /kaggle/working /kaggle/input 2>/dev/null | head -5

In [ ]:
import os, glob, yaml
SRC = '/kaggle/input/mega-dataset-dental'
# najdi skutocny koren datasetu (moze byt nested)
hits = glob.glob(f'{SRC}/**/data.yaml', recursive=True)
print('data.yaml najdene:', hits)
root = os.path.dirname(hits[0])
print('dataset root:', root)
for split in ['train/images','valid/images','test/images']:
    p = os.path.join(root, split)
    print(split, len(os.listdir(p)) if os.path.isdir(p) else 'CHYBA')

In [ ]:
# Uprav data.yaml: absolutne cesty + valid split sa vola 'valid'
cfg = yaml.safe_load(open(os.path.join(root,'data.yaml')))
cfg['path'] = root
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
os.makedirs('/kaggle/working/ds', exist_ok=True)
yaml.safe_dump(cfg, open('/kaggle/working/ds/data.yaml','w'), sort_keys=False)
print(open('/kaggle/working/ds/data.yaml').read())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
results = model.train(
    data='/kaggle/working/ds/data.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    cache='ram',          # 765 MB sa vojde do RAM -> zrychli a setri disk
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    # augmentacia zamerana na caries (slaba trieda)
    hsv_v=0.5,            # silnejsia jasova augmentacia (RTG kontrast)
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    project='/kaggle/working/train',
    name='v1_yolov8s_mega',
    exist_ok=True,
)

In [ ]:
# Validacia best modelu na test sete
best = YOLO('/kaggle/working/train/v1_yolov8s_mega/weights/best.pt')
metrics = best.val(data='/kaggle/working/ds/data.yaml', split='test')
print(metrics.results_dict)

In [ ]:
# Presun do working root ako kernel output + uprac velke nepodstatne subory
import shutil, os
shutil.copy('/kaggle/working/train/v1_yolov8s_mega/weights/best.pt', '/kaggle/working/best_v1.pt')
shutil.copy('/kaggle/working/train/v1_yolov8s_mega/weights/last.pt', '/kaggle/working/last_v1.pt')
# results.csv pre analyzu
shutil.copy('/kaggle/working/train/v1_yolov8s_mega/results.csv', '/kaggle/working/results_v1.csv')
shutil.rmtree('/kaggle/working/train', ignore_errors=True)
shutil.rmtree('/kaggle/working/ds', ignore_errors=True)
!ls -la /kaggle/working
!df -h /kaggle/working